# Stream 3 Tabular Risk Model Benchmark

## 1. Purpose & Protocol
This notebook documents and reproduces the comparative benchmark of machine learning model families for fetal risk prediction from maternal clinical characteristics (CDC Natality + Stillbirths dataset).

All models are evaluated under the exact same strict protocol:
- **Temporal Split**: Training cohort with births <= 2021, held-out test cohort with births >= 2022.
- **Gestation Cohort**: Gestational age >= 20 weeks (`COMBGEST >= 20`).
- **Target Settings**:
  - **E1 (Primary Features)**: 15 maternal risk factors and clinical indicators.
  - **E4 (Clinical-Only Features)**: 13 strictly pre-delivery clinical features (no gestational age / sex).
- **Models Compared**:
  1. **Logistic Regression**: Median numeric imputation + StandardScaler, categorical mode + OneHotEncoder, class_weight='balanced'.
  2. **Random Forest**: 100 estimators, max_depth=16 (tuned strictly on internal validation split).
  3. **LightGBM**: 300 estimators, max_depth=5, learning_rate=0.05.
  4. **Our XGBoost**: Exact repository pipeline reproducing historical benchmark results.
- **Verification Criteria**:
  - Test AUC on E1 must equal **0.738** (reproduced: 0.7381, diff <= 0.0001).
  - Test AUC on E4 must equal **0.707** (reproduced: 0.7070, diff: 0.0000).


In [1]:
import sys
from pathlib import Path
import subprocess

# Locate work root
work_dir = Path.cwd()
while work_dir.name != "work" and work_dir.parent != work_dir:
    work_dir = work_dir.parent
if str(work_dir / "src") not in sys.path:
    sys.path.insert(0, str(work_dir / "src"))

print(f"Work directory: {work_dir}")
script_path = work_dir / "scripts" / "18_bench_stream3.py"
print(f"Executing {script_path.name} on CPU...")

proc = subprocess.run(
    [sys.executable, str(script_path), "--n-boot", "200"],
    cwd=str(work_dir),
    capture_output=True,
    text=True,
    check=True,
)
print(proc.stdout)


Work directory: C:\fetal-github-clone\work
Executing 18_bench_stream3.py on CPU...


Stream 3 Clinical Tabular Risk Model Benchmark (Comparison Study)
CSV Path:       C:\fetal-data\cdc\clinical_training_data.csv
Out Directory:  C:\fetal-github-clone\work\docs\results\stream3_bench
LightGBM ready: True
Smoke Mode:     False

[Step 1] Loading and cleaning dataset...
  Loaded raw: (718136, 80) | Deduped: (714973, 80) (Removed: 3163)
  Gestation >= 20 weeks cohort: 532392 rows

--- Setting: E1 (Primary, >=20 wk) ---
  Train: 402282 rows (132604 stillbirths) | Test: 130110 rows (40221 stillbirths)
  Features (15): ['MAGER', 'BMI', 'PRECARE', 'PRIORDEAD', 'PRIORLIVE', 'RF_GDIAB', 'RF_GHYPE', 'RF_EHYPE', 'RF_CESAR', 'RF_ARTEC', 'RF_INFTR', 'RF_FEDRG', 'CIG_REC', 'DPLURAL', 'SEX']
  Logistic Regression    | AUC: 0.6659 [95% CI: 0.6629, 0.6691] | Fit: 5.7s
  Random Forest          | AUC: 0.7322 [95% CI: 0.7297, 0.7354] | Fit: 50.5s
  LightGBM               | AUC: 0.7380 [95% CI: 0.7357, 0.7413] | Fit: 10.8s
  Our XGBoost            | AUC: 0.7381 [95% CI: 0.7358, 0.7413] | Fit: 

In [2]:
import json
from pathlib import Path
import pandas as pd

table_file = work_dir / "docs" / "results" / "stream3_bench" / "benchmark_table.md"
results_file = work_dir / "docs" / "results" / "stream3_bench" / "benchmark_results.json"

with open(results_file, "r", encoding="utf-8") as f:
    bench_data = json.load(f)

print("=== Stream 3 Comparison Benchmark Results Table ===\n")
print(table_file.read_text(encoding="utf-8"))

print("\n=== XGBoost Reproduction Check vs Historical Targets ===")
xgb_e1 = next(r for r in bench_data["results"] if r["setting"].startswith("E1") and r["model"] == "Our XGBoost")
xgb_e4 = next(r for r in bench_data["results"] if r["setting"].startswith("E4") and r["model"] == "Our XGBoost")

e1_diff = abs(round(xgb_e1["test_auc"], 3) - 0.738)
e4_diff = abs(round(xgb_e4["test_auc"], 3) - 0.707)

print(f"E1 XGBoost Test AUC: {xgb_e1['test_auc']:.4f} (Target: 0.738, Diff: {e1_diff:.4f}) -> Pass: {e1_diff <= 1e-4}")
print(f"E4 XGBoost Test AUC: {xgb_e4['test_auc']:.4f} (Target: 0.707, Diff: {e4_diff:.4f}) -> Pass: {e4_diff <= 1e-4}")


=== Stream 3 Comparison Benchmark Results Table ===

# Stream 3 Clinical Risk Model Benchmark

Comparison of model families under the exact same protocol: CDC natality/stillbirth data, temporal split (Train <= 2021, Test >= 2022), no post-delivery leakage.

| Setting | Model | Train Rows | Test Rows | Test AUC | 95% Bootstrap CI | Fit Time (s) |
|:---|:---|---:|---:|---:|:---:|---:|
| E1 (Primary, >=20 wk) | Logistic Regression | 402,282 | 130,110 | **0.6659** | [0.6629, 0.6691] | 5.7 |
| E1 (Primary, >=20 wk) | Random Forest | 402,282 | 130,110 | **0.7322** | [0.7297, 0.7354] | 50.5 |
| E1 (Primary, >=20 wk) | LightGBM | 402,282 | 130,110 | **0.7380** | [0.7357, 0.7413] | 10.8 |
| E1 (Primary, >=20 wk) | Our XGBoost | 402,282 | 130,110 | **0.7381** | [0.7358, 0.7413] | 22.6 |
| E4 (Clinical-Only, >=20 wk) | Logistic Regression | 402,282 | 130,110 | **0.6585** | [0.6557, 0.6617] | 4.8 |
| E4 (Clinical-Only, >=20 wk) | Random Forest | 402,282 | 130,110 | **0.7043** | [0.7013, 0.7075] | 

In [3]:
import time
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from fetal import config
from fetal.clinical.data import load_clean
from fetal.clinical.constants import FEATURE_SETS

print("=== Random Forest Validation Search for max_depth ===")
print("Protocol: Train <= 2019, Validation 2020-2021 (Test >= 2022 strictly untouched)\n")

raw, df = load_clean(config.CLINICAL_CSV)
d20 = df[df.COMBGEST >= 20].copy()

tr_sub = d20[d20.YEAR <= 2019].copy()
val_sub = d20[(d20.YEAR >= 2020) & (d20.YEAR <= 2021)].copy()

num = FEATURE_SETS["primary"]["num"]
cat = FEATURE_SETS["primary"]["cat"]
X_tr, y_tr = tr_sub[num + cat], tr_sub.OUTCOME.values
X_val, y_val = val_sub[num + cat], val_sub.OUTCOME.values

print(f"Training slice (<= 2019)  : {len(tr_sub):,} rows ({y_tr.sum():,} positive)")
print(f"Validation slice (2020-21): {len(val_sub):,} rows ({y_val.sum():,} positive)\n")

parts = [("n", SimpleImputer(strategy="median"), num)]
if cat:
    parts.append(("c", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat))

pre = ColumnTransformer(parts)
X_tr_proc = pre.fit_transform(X_tr)
X_val_proc = pre.transform(X_val)

depth_rows = []
for d in [8, 10, 12, 14, 16, 18, 20]:
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42, n_jobs=-1)
    rf.fit(X_tr_proc, y_tr)
    p_val = rf.predict_proba(X_val_proc)[:, 1]
    auc = roc_auc_score(y_val, p_val)
    elapsed = time.time() - t0
    depth_rows.append({"max_depth": d, "val_auc": round(auc, 5), "fit_time_sec": round(elapsed, 1)})
    print(f"Depth {d:2d} | Val AUC: {auc:.5f} | Fit time: {elapsed:.1f}s")

df_depths = pd.DataFrame(depth_rows)
print("\nValidation Search Summary Table:")
print(df_depths.to_string(index=False))

best_row = max(depth_rows, key=lambda r: r["val_auc"])
print(f"\nOptimal max_depth selected on internal validation: max_depth={best_row['max_depth']} (Val AUC: {best_row['val_auc']:.5f})")
print("This hyperparameter was frozen and used for the final test benchmark without test set peeking.")


=== Random Forest Validation Search for max_depth ===
Protocol: Train <= 2019, Validation 2020-2021 (Test >= 2022 strictly untouched)



Training slice (<= 2019)  : 270,353 rows (90,568 positive)
Validation slice (2020-21): 131,929 rows (42,036 positive)



Depth  8 | Val AUC: 0.71827 | Fit time: 17.3s


Depth 10 | Val AUC: 0.72119 | Fit time: 21.5s


Depth 12 | Val AUC: 0.72313 | Fit time: 22.9s


Depth 14 | Val AUC: 0.72418 | Fit time: 24.8s


Depth 16 | Val AUC: 0.72547 | Fit time: 27.7s


Depth 18 | Val AUC: 0.72545 | Fit time: 28.5s


Depth 20 | Val AUC: 0.72460 | Fit time: 31.0s

Validation Search Summary Table:
 max_depth  val_auc  fit_time_sec
         8  0.71827          17.3
        10  0.72119          21.5
        12  0.72313          22.9
        14  0.72418          24.8
        16  0.72547          27.7
        18  0.72545          28.5
        20  0.72460          31.0

Optimal max_depth selected on internal validation: max_depth=16 (Val AUC: 0.72547)
This hyperparameter was frozen and used for the final test benchmark without test set peeking.
